<a href="https://colab.research.google.com/github/nadjmif/K1_StatsDeskriptif/blob/main/Statistik_dan_probalitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# -*- coding: utf-8 -*-
"""
STATISTIKA DESKRIPTIF & EKSPLORASI VISUAL — PT Bayan Resources Tbk (BYAN)
=========================================================================
Cakupan : statistika deskriptif + eksplorasi visual (TIDAK ada uji hipotesis,
          fitting distribusi, model deret waktu, atau event study formal).
Input   : 3TIC_01_DataHistoris_EventRumorBYAN.xlsx
          - BYAN_Analysis : Date, Open, High, Low, Close, Volume, Return,
                            %ΔVolume, Event, Event_ID, Periode_Event, Rumor_Event
          - Event_Rumor   : Event_ID, Tanggal, Isi_Topik_Rumor,
                            Status_Klarifikasi, 'Rumor (1=Rumor, 0=Non-Rumor)',
                            Nomor_Surat, Baris_Data_BYAN
Output  : outputs/  (CSV + 1 Excel ringkasan)   figures/ (PNG, 300 dpi)

Catatan satuan:
  - Return disimpan sebagai PROPORSI (0,01 = 1%); pada grafik ditampilkan dalam %.
  - Volume dalam LOT (1 lot = 100 lembar), sesuai sheet Skala_Variabel.
"""

# ==========================================================================
# 00. SETUP
# ==========================================================================
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
from scipy import stats

# --- Konfigurasi file ------------------------------------------------------
INPUT_FILE = Path("3TIC_01_DataHistoris_EventRumorBYAN.xlsx")
OUT_DIR = Path("outputs")
FIG_DIR = Path("figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# --- Nama sheet & kolom AKTUAL (hasil pemeriksaan workbook) -----------------
SHEET_DATA = "BYAN_Analysis"
SHEET_EVENT = "Event_Rumor"
COL_RUMOR_EVENT_SHEET = "Rumor (1=Rumor, 0=Non-Rumor)"   # nama kolom di Event_Rumor
PERIOD_ORDER = ["Normal", "Pre", "Event", "Post"]        # label yang ada di dataset

# --- Opsi tampilan / metode -------------------------------------------------
# Skewness & kurtosis: True = koreksi sampel (sama dengan SKEW() dan KURT() Excel),
# False = momen populasi. Sesuaikan dengan yang dipakai pada laporan.
BIAS_CORRECTED = True
FIG_DPI = 300
HIST_BINS = 50
# Ukuran untuk fig_4_7: "median_volume" atau "mean_absret"
SUMMARY_MEASURE = "median_volume"
# Grafik event individual (fig_4_8) bersifat opsional; ubah ke False jika tidak dibahas
MAKE_EVENT_FIGURE = True

# Tidak ada proses acak pada analisis ini -> TIDAK ada random seed.

# Warna: merah hanya untuk membedakan rumor; selain itu abu-abu/hitam
C_RUMOR = "#C00000"
C_NONRUMOR = "0.35"

plot_counts = {}   # jumlah observasi yang dipakai tiap grafik (untuk validasi)


def fmt_id(x, pos=None):
    """Format ribuan gaya Indonesia (titik sebagai pemisah ribuan)."""
    return f"{x:,.0f}".replace(",", ".")


def require_columns(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Kolom {missing} tidak ditemukan pada sheet '{name}'. "
            f"Kolom yang tersedia: {df.columns.tolist()}"
        )


# ==========================================================================
# 01. LOAD DATA
# ==========================================================================
def load_data(path):
    xl = pd.ExcelFile(path)
    print("Sheet pada workbook:", xl.sheet_names)
    for s in (SHEET_DATA, SHEET_EVENT):
        if s not in xl.sheet_names:
            raise KeyError(f"Sheet '{s}' tidak ada. Sheet tersedia: {xl.sheet_names}")
    raw_data = xl.parse(SHEET_DATA)
    raw_event = xl.parse(SHEET_EVENT)

    for name, d in ((SHEET_DATA, raw_data), (SHEET_EVENT, raw_event)):
        print("\n" + "=" * 70)
        print(f"SHEET: {name}")
        print("=" * 70)
        print("shape :", d.shape)
        print("kolom :", d.columns.tolist())
        print("\n5 baris pertama:")
        print(d.head().to_string())
        print("\nTipe data:")
        print(d.dtypes.to_string())
        print("\nMissing values:")
        print(d.isna().sum().to_string())
    return raw_data, raw_event


# ==========================================================================
# 02. DATA PREPARATION
# ==========================================================================
def prepare_data(raw_data, raw_event):
    require_columns(raw_data, ["Date", "Close", "Volume", "Event_ID", "Periode_Event"], SHEET_DATA)
    require_columns(raw_event, ["Event_ID", "Tanggal", COL_RUMOR_EVENT_SHEET], SHEET_EVENT)

    df = raw_data.copy()
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce")
    if not df["Date"].is_monotonic_increasing:
        print("PERINGATAN: Date tidak terurut -> diurutkan menaik.")
        df = df.sort_values("Date").reset_index(drop=True)

    # Return_t = (Close_t - Close_{t-1}) / Close_{t-1}
    ret_calc = (df["Close"] - df["Close"].shift(1)) / df["Close"].shift(1)
    if "Return" in df.columns and pd.to_numeric(df["Return"], errors="coerce").notna().any():
        df["Return"] = pd.to_numeric(df["Return"], errors="coerce")
        print("\nReturn: memakai kolom 'Return' bawaan dataset (diverifikasi pada bagian 14).")
    else:
        df["Return"] = ret_calc
        print("\nReturn: kolom tidak tersedia -> dihitung dari Close.")
    df.attrs["return_recalc"] = ret_calc

    df["absRet"] = df["Return"].abs()
    # ln(Volume) hanya untuk Volume > 0; nol tidak diganti angka arbitrer (menjadi NaN)
    df["logVol"] = np.log(df["Volume"].where(df["Volume"] > 0))

    # Label periode dan rumor
    unknown = set(df["Periode_Event"].dropna().unique()) - set(PERIOD_ORDER)
    if unknown:
        print(f"PERINGATAN: label periode di luar {PERIOD_ORDER}: {unknown}")
    df["Periode_Event"] = pd.Categorical(df["Periode_Event"], categories=PERIOD_ORDER, ordered=True)

    if "Rumor_Event" in df.columns:
        # Normal bernilai '-' -> NaN (tidak berlabel rumor/non-rumor)
        df["Rumor_Num"] = pd.to_numeric(df["Rumor_Event"], errors="coerce")
        df["Rumor_Label"] = df["Rumor_Num"].map({1: "Rumor", 0: "Non-Rumor"})

    ev = raw_event.rename(columns={COL_RUMOR_EVENT_SHEET: "Rumor"}).copy()
    ev["Tanggal"] = pd.to_datetime(ev["Tanggal"], errors="coerce")
    ev["Rumor_Label"] = ev["Rumor"].map({1: "Rumor", 0: "Non-Rumor"})

    print("\n--- Ringkasan persiapan data ---")
    print("Jumlah observasi              :", len(df))
    print("Date tidak valid              :", int(df["Date"].isna().sum()))
    print("Close missing                 :", int(df["Close"].isna().sum()))
    print("Volume missing                :", int(df["Volume"].isna().sum()))
    print("Return missing                :", int(df["Return"].isna().sum()), "(baris pertama tidak punya hari sebelumnya)")
    print("Return valid                  :", int(df["Return"].notna().sum()))
    print("Return = 0                    :", int((df["Return"] == 0).sum()))
    print("Volume = 0 (ln tidak dihitung):", int((df["Volume"] == 0).sum()))
    print("logVol valid                  :", int(df["logVol"].notna().sum()))
    return df, ev


# ==========================================================================
# 03. STATISTIK DESKRIPTIF KESELURUHAN
# ==========================================================================
def describe_series(x):
    x = pd.Series(x).dropna()
    n = len(x)
    return {
        "n": n,
        "mean": x.mean(),
        "median": x.median(),
        "sd": x.std(ddof=1),
        "min": x.min(),
        "max": x.max(),
        "skewness": stats.skew(x, bias=not BIAS_CORRECTED) if n > 2 else np.nan,
        "excess_kurtosis": stats.kurtosis(x, fisher=True, bias=not BIAS_CORRECTED) if n > 3 else np.nan,
    }


def table_overall(df):
    rows = {v: describe_series(df[v]) for v in ["Return", "Volume", "absRet"]}
    t = pd.DataFrame(rows).T
    t["n"] = t["n"].astype(int)
    t.index.name = "Variabel"
    return t


# ==========================================================================
# 04. STATISTIK DESKRIPTIF PERIODE
# ==========================================================================
def table_period(df):
    g = df.groupby("Periode_Event", observed=True)
    t = pd.DataFrame({
        "n_obs": g.size(),
        "n_return_valid": g["Return"].count(),
        "mean_Return": g["Return"].mean(),
        "median_Return": g["Return"].median(),
        "sd_Return": g["Return"].std(ddof=1),
        "mean_absRet": g["absRet"].mean(),
        "median_Volume": g["Volume"].median(),
        "median_logVol": g["logVol"].median(),
    }).reindex(PERIOD_ORDER)
    t.index.name = "Periode_Event"
    return t


# ==========================================================================
# 05. STATISTIK DESKRIPTIF RUMOR / NON-RUMOR  (deskriptif saja, tanpa uji)
# ==========================================================================
def table_rumor(df):
    if "Rumor_Label" not in df.columns:
        print("Kolom Rumor_Event tidak tersedia -> bagian 05 dilewati.")
        return None
    sub = df[df["Rumor_Label"].notna()]       # hanya observasi berlabel (Pre/Event/Post)
    g = sub.groupby("Rumor_Label")
    t = pd.DataFrame({
        "n_obs": g.size(),
        "n_return_valid": g["Return"].count(),
        "mean_Return": g["Return"].mean(),
        "sd_Return": g["Return"].std(ddof=1),
        "mean_absRet": g["absRet"].mean(),
        "median_Volume": g["Volume"].median(),
    }).reindex(["Rumor", "Non-Rumor"])
    t.index.name = "Kelompok"
    return t


# ==========================================================================
# 06. ANALISIS EVENT INDIVIDUAL
# ==========================================================================
def table_per_event(df, ev):
    """
    Kunci gabung: Event_ID. Hari event = baris dengan Periode_Event == 'Event'.
    Jendela event = semua baris ber-Event_ID yang sama (Pre + Event + Post
    sebagaimana dilabeli dataset). Karena jendela event berdekatan (mis. E05-E06,
    E07-E08) dipotong oleh dataset, n_pre / n_post dilaporkan per event.
    """
    med_normal = df.loc[df["Periode_Event"] == "Normal", "Volume"].median()
    day = df[df["Periode_Event"] == "Event"].set_index("Event_ID")
    rows = []
    for _, r in ev.iterrows():
        eid = r["Event_ID"]
        if eid not in day.index:
            raise KeyError(f"Event_ID {eid} tidak memiliki baris 'Event' pada {SHEET_DATA}")
        d = day.loc[eid]
        win = df[df["Event_ID"] == eid]
        rows.append({
            "Event_ID": eid,
            "Tanggal": r["Tanggal"].date(),
            "Rumor_Label": r["Rumor_Label"],
            "Return_hari_event": d["Return"],
            "Volume_hari_event": d["Volume"],
            "Median_Volume_Normal": med_normal,
            "Rasio_Volume_vs_Median_Normal": d["Volume"] / med_normal,
            "SD_Return_jendela": win["Return"].std(ddof=1),
            "n_jendela": len(win),
            "n_pre": int((win["Periode_Event"] == "Pre").sum()),
            "n_post": int((win["Periode_Event"] == "Post").sum()),
        })
    return pd.DataFrame(rows)


# ==========================================================================
# 07-13. VISUALISASI
# ==========================================================================
def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print("  tersimpan:", path)


def fig_time_series(df, ev):
    """07. Close & Volume vs Date + garis vertikal tanggal event (tanpa klaim kausal)."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                                   gridspec_kw={"height_ratios": [3, 2]})
    ax1.plot(df["Date"], df["Close"], color="black", lw=1.0)
    ax2.plot(df["Date"], df["Volume"], color="black", lw=0.8)
    for _, r in ev.iterrows():
        rumor = r["Rumor"] == 1
        for ax in (ax1, ax2):
            ax.axvline(r["Tanggal"], color=C_RUMOR if rumor else C_NONRUMOR,
                       ls="-" if rumor else "--", lw=1.0, alpha=0.85)
    ax1.set_ylabel("Harga penutupan (Rp)")
    ax2.set_ylabel("Volume (lot)")
    ax2.set_xlabel("Tanggal")
    ax1.yaxis.set_major_formatter(FuncFormatter(fmt_id))
    ax2.yaxis.set_major_formatter(FuncFormatter(fmt_id))
    ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax1.legend(handles=[
        Line2D([0], [0], color="black", lw=1.0, label="Close"),
        Line2D([0], [0], color=C_RUMOR, ls="-", label="Tanggal event: rumor"),
        Line2D([0], [0], color=C_NONRUMOR, ls="--", label="Tanggal event: non-rumor"),
    ], loc="lower left", frameon=False)
    ax1.set_title("Harga Penutupan dan Volume Harian BYAN")
    fig.text(0.01, -0.01, "Garis vertikal menandai tanggal event pada Event_Rumor; "
             "tidak menunjukkan hubungan sebab-akibat.", fontsize=8, ha="left")
    fig.tight_layout()
    plot_counts["fig_4_1_close"] = int(df["Close"].notna().sum())
    plot_counts["fig_4_1_volume"] = int(df["Volume"].notna().sum())
    save_fig(fig, "fig_4_1_harga_volume.png")


def fig_hist_return(df):
    """08. Histogram Return empiris (tanpa kurva teoritis)."""
    x = df["Return"].dropna() * 100
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.hist(x, bins=HIST_BINS, color="0.65", edgecolor="0.25", lw=0.5)
    ax.axvline(0, color="black", ls="--", lw=1.0, label="Return = 0")
    ax.set_xlabel("Return harian (%)")
    ax.set_ylabel("Frekuensi (jumlah hari)")
    ax.set_title(f"Histogram Return Harian BYAN (n = {len(x)})")
    ax.legend(frameon=False)
    fig.tight_layout()
    plot_counts["fig_4_2"] = len(x)
    save_fig(fig, "fig_4_2_hist_return.png")


def fig_hist_volume(df):
    """09a. Histogram Volume pada skala asli (lot)."""
    x = df["Volume"].dropna()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.hist(x, bins=HIST_BINS, color="0.65", edgecolor="0.25", lw=0.5)
    ax.set_xlabel("Volume (lot)")
    ax.set_ylabel("Frekuensi (jumlah hari)")
    ax.xaxis.set_major_formatter(FuncFormatter(fmt_id))
    ax.set_title(f"Histogram Volume Harian BYAN, skala asli (n = {len(x)})")
    fig.text(0.01, -0.02, "Skala asli: menunjukkan kondisi sebenarnya, termasuk nilai volume ekstrem.",
             fontsize=8, ha="left")
    fig.tight_layout()
    plot_counts["fig_4_3"] = len(x)
    save_fig(fig, "fig_4_3_hist_volume.png")


def fig_hist_logvolume(df):
    """09b. Histogram ln(Volume): hanya membantu membaca struktur distribusi."""
    x = df["logVol"].dropna()
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.hist(x, bins=HIST_BINS, color="0.65", edgecolor="0.25", lw=0.5)
    ax.set_xlabel("ln(Volume)  [Volume dalam lot; skala logaritma natural]")
    ax.set_ylabel("Frekuensi (jumlah hari)")
    ax.set_title(f"Histogram ln(Volume) Harian BYAN (n = {len(x)})")
    fig.text(0.01, -0.02, "Transformasi ln hanya mengubah skala agar struktur distribusi lebih mudah dibaca; "
             "observasi ekstrem tetap ada.", fontsize=8, ha="left")
    fig.tight_layout()
    plot_counts["fig_4_4"] = len(x)
    save_fig(fig, "fig_4_4_hist_logvolume.png")


def boxplot_by_period(df, col, scale, ylabel, title, fname, key, zero_line=False):
    """10-11. Boxplot per periode (outlier tidak dipotong)."""
    data, labels, counts = [], [], {}
    for p in PERIOD_ORDER:
        v = df.loc[df["Periode_Event"] == p, col].dropna().to_numpy() * scale
        data.append(v)
        labels.append(f"{p}\n(n = {len(v)})")
        counts[p] = len(v)
    fig, ax = plt.subplots(figsize=(8, 6))
    bp = ax.boxplot(data, patch_artist=True, showfliers=True, widths=0.55,
                    medianprops=dict(color="black", lw=1.6),
                    flierprops=dict(marker="o", markersize=3.5, markerfacecolor="none",
                                    markeredgecolor="0.3"))
    for b in bp["boxes"]:
        b.set(facecolor="0.85", edgecolor="0.2")
    ax.set_xticks(range(1, len(PERIOD_ORDER) + 1))
    ax.set_xticklabels(labels)
    if zero_line:
        ax.axhline(0, color="black", ls=":", lw=0.9)
    ax.set_xlabel("Periode")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    fig.text(0.01, -0.02, "Kotak: Q1-Q3; garis tengah: median; whisker: 1,5 x IQR; "
             "lingkaran: observasi di luar whisker.", fontsize=8, ha="left")
    fig.tight_layout()
    plot_counts[key] = counts
    save_fig(fig, fname)


def fig_summary(tbl_period):
    """12. Ringkasan antar periode: satu ukuran, diambil dari tabel periode yang sama."""
    if SUMMARY_MEASURE == "median_volume":
        col, scale, ylabel = "median_Volume", 1, "Median Volume (lot)"
        title = "Median Volume Harian per Periode"
    elif SUMMARY_MEASURE == "mean_absret":
        col, scale, ylabel = "mean_absRet", 100, "Rata-rata |Return| harian (%)"
        title = "Rata-rata |Return| Harian per Periode"
    else:
        raise ValueError("SUMMARY_MEASURE harus 'median_volume' atau 'mean_absret'")
    vals = tbl_period[col] * scale
    fig, ax = plt.subplots(figsize=(8, 5.5))
    bars = ax.bar(PERIOD_ORDER, vals.to_numpy(), color="0.65", edgecolor="0.2", width=0.6)
    for b, v in zip(bars, vals):
        txt = fmt_id(v) if SUMMARY_MEASURE == "median_volume" else f"{v:.3f}".replace(".", ",")
        ax.annotate(txt, (b.get_x() + b.get_width() / 2, b.get_height()),
                    ha="center", va="bottom", fontsize=9, xytext=(0, 3), textcoords="offset points")
    ax.set_xticks(range(len(PERIOD_ORDER)))
    ax.set_xticklabels([f"{p}\n(n = {int(tbl_period.loc[p, 'n_obs'])})" for p in PERIOD_ORDER])
    ax.set_xlabel("Periode")
    ax.set_ylabel(ylabel)
    if SUMMARY_MEASURE == "median_volume":
        ax.yaxis.set_major_formatter(FuncFormatter(fmt_id))
    ax.set_title(title)
    fig.tight_layout()
    plot_counts["fig_4_7"] = vals.round(12).to_dict()
    save_fig(fig, "fig_4_7_ringkasan_periode.png")


def fig_event_ratio(tbl_event):
    """13 (opsional). Rasio volume hari event terhadap median volume Normal."""
    fig, ax = plt.subplots(figsize=(9, 5.5))
    colors = [C_RUMOR if x == "Rumor" else C_NONRUMOR for x in tbl_event["Rumor_Label"]]
    ax.bar(range(len(tbl_event)), tbl_event["Rasio_Volume_vs_Median_Normal"], color=colors,
           edgecolor="0.2", width=0.6)
    ax.axhline(1, color="black", ls="--", lw=1.0)
    ax.set_xticks(range(len(tbl_event)))
    ax.set_xticklabels([f"{e}\n{pd.Timestamp(t):%d/%m/%Y}"
                        for e, t in zip(tbl_event["Event_ID"], tbl_event["Tanggal"])], fontsize=8)
    ax.set_xlabel("Event")
    ax.set_ylabel("Rasio Volume hari event / median Volume Normal (kali)")
    ax.set_title("Rasio Volume Hari Event terhadap Median Volume Periode Normal")
    ax.legend(handles=[Patch(facecolor=C_RUMOR, edgecolor="0.2", label="Rumor"),
                       Patch(facecolor=C_NONRUMOR, edgecolor="0.2", label="Non-rumor"),
                       Line2D([0], [0], color="black", ls="--", label="Rasio = 1 (median Normal)")],
              frameon=False)
    fig.tight_layout()
    plot_counts["fig_4_8"] = len(tbl_event)
    save_fig(fig, "fig_4_8_rasio_volume_event.png")


# ==========================================================================
# 14. VALIDATION / CONSISTENCY CHECK
# ==========================================================================
def _manual_skew_kurt(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    m = x.mean()
    m2, m3, m4 = [((x - m) ** k).mean() for k in (2, 3, 4)]
    g1, g2 = m3 / m2 ** 1.5, m4 / m2 ** 2 - 3
    if not BIAS_CORRECTED:
        return g1, g2
    G1 = np.sqrt(n * (n - 1)) / (n - 2) * g1
    G2 = (n - 1) / ((n - 2) * (n - 3)) * ((n + 1) * g2 + 6)
    return G1, G2


def validate(df, ev, raw_data, raw_event, t_overall, t_period, t_rumor, t_event):
    checks = []

    def add(name, ok, detail=""):
        checks.append((name, bool(ok), detail))

    # jumlah data sesuai Excel
    try:
        import openpyxl
        wb = openpyxl.load_workbook(INPUT_FILE, read_only=True)
        n_excel = sum(1 for r in wb[SHEET_DATA].iter_rows(min_row=2, max_col=1, values_only=True)
                      if r[0] is not None)
        n_excel_ev = sum(1 for r in wb[SHEET_EVENT].iter_rows(min_row=2, max_col=1, values_only=True)
                         if r[0] is not None)
        wb.close()
        add("Jumlah baris data = baris Excel", len(df) == n_excel, f"{len(df)} vs {n_excel}")
        add("Jumlah event = baris sheet Event_Rumor", len(ev) == n_excel_ev, f"{len(ev)} vs {n_excel_ev}")
    except Exception as e:  # noqa
        add("Jumlah baris data = baris Excel", len(df) == len(raw_data), f"(openpyxl tidak dipakai: {e})")

    # Return
    n_ret_expected = int(df.attrs["return_recalc"].notna().sum())
    add("Jumlah Return valid = jumlah Return hitung ulang", df["Return"].notna().sum() == n_ret_expected,
        f"{int(df['Return'].notna().sum())} vs {n_ret_expected}")
    diff = (df["Return"] - df.attrs["return_recalc"]).abs().max()
    add("Return bawaan = (Close_t - Close_t-1)/Close_t-1", diff < 1e-9, f"selisih maks {diff:.2e}")

    # Event
    add("Event_ID unik pada Event_Rumor", ev["Event_ID"].is_unique)
    ids_data = set(df["Event_ID"].astype(str)) - {"-"}
    add("Semua Event_ID cocok antar sheet", ids_data == set(ev["Event_ID"]),
        f"data={sorted(ids_data)}")
    ev_dates = set(ev["Tanggal"])
    day_dates = set(df.loc[df["Periode_Event"] == "Event", "Date"])
    add("Tanggal event valid & ada di BYAN_Analysis", ev["Tanggal"].notna().all() and ev_dates == day_dates)
    add("Jumlah hari 'Event' = jumlah event", (df["Periode_Event"] == "Event").sum() == len(ev))
    if "Baris_Data_BYAN" in ev.columns:   # nomor baris Excel (header = baris 1)
        pos = ev["Tanggal"].map(lambda d: int(df.index[df["Date"] == d][0]) + 2)
        add("Baris_Data_BYAN = nomor baris Excel tanggal event", (pos == ev["Baris_Data_BYAN"]).all())
    if "Rumor_Num" in df.columns:
        flag_data = df[df["Rumor_Num"].notna()].groupby("Event_ID")["Rumor_Num"].first()
        flag_ev = ev.set_index("Event_ID")["Rumor"]
        add("Status rumor konsisten antar sheet", (flag_data.reindex(flag_ev.index) == flag_ev).all())

    # Periode
    add("Semua label periode valid", df["Periode_Event"].notna().all())
    add("n Normal+Pre+Event+Post = n data", t_period["n_obs"].sum() == len(df))
    add("Event_ID '-' hanya pada periode Normal",
        ((df["Event_ID"].astype(str) == "-") == (df["Periode_Event"] == "Normal")).all())

    # Infinite / log(0)
    num = df.select_dtypes(include=[np.number])
    add("Tidak ada nilai infinite", not np.isinf(num.to_numpy(dtype=float)).any())
    add("Volume > 0 (ln tidak bermasalah)", (df["Volume"] > 0).all())
    add("logVol valid = n Volume", df["logVol"].notna().sum() == df["Volume"].notna().sum())

    # Reproduksi tabel secara independen
    ok = True
    for var in ["Return", "Volume", "absRet"]:
        x = df[var].dropna().to_numpy(dtype=float)
        sk, ku = _manual_skew_kurt(x)
        ref = [len(x), x.mean(), np.median(x), x.std(ddof=1), x.min(), x.max(), sk, ku]
        got = t_overall.loc[var].to_numpy(dtype=float)
        ok &= np.allclose(ref, got, rtol=1e-9, atol=1e-12)
    add("Tabel statistik deskriptif keseluruhan dapat direproduksi", ok)

    ok = True
    for p in PERIOD_ORDER:
        s = df[df["Periode_Event"] == p]
        ref = [len(s), s["Return"].notna().sum(), np.nanmean(s["Return"]), np.nanmedian(s["Return"]),
               np.nanstd(s["Return"], ddof=1), np.nanmean(np.abs(s["Return"])),
               np.median(s["Volume"]), np.nanmedian(np.log(s["Volume"]))]
        ok &= np.allclose(ref, t_period.loc[p].to_numpy(dtype=float), rtol=1e-9, atol=1e-12)
    add("Tabel periode dapat direproduksi", ok)

    if t_rumor is not None:
        ok = True
        for k in ["Rumor", "Non-Rumor"]:
            s = df[df["Rumor_Label"] == k]
            ref = [len(s), s["Return"].notna().sum(), np.nanmean(s["Return"]),
                   np.nanstd(s["Return"], ddof=1), np.nanmean(np.abs(s["Return"])), np.median(s["Volume"])]
            ok &= np.allclose(ref, t_rumor.loc[k].to_numpy(dtype=float), rtol=1e-9, atol=1e-12)
        add("Tabel rumor/non-rumor dapat direproduksi", ok)

    # Grafik memakai data yang sama dengan tabel
    add("Histogram Return: n = n Return valid", plot_counts.get("fig_4_2") == t_overall.loc["Return", "n"])
    add("Histogram Volume: n = n Volume", plot_counts.get("fig_4_3") == t_overall.loc["Volume", "n"])
    add("Histogram ln(Volume): n = n Volume", plot_counts.get("fig_4_4") == t_overall.loc["Volume", "n"])
    add("Boxplot Return: n per periode = tabel periode",
        plot_counts.get("box_return") == t_period["n_return_valid"].astype(int).to_dict())
    add("Boxplot ln(Volume): n per periode = tabel periode",
        plot_counts.get("box_logvol") == t_period["n_obs"].astype(int).to_dict())
    add("Time series memakai seluruh observasi",
        plot_counts.get("fig_4_1_close") == len(df) and plot_counts.get("fig_4_1_volume") == len(df))

    print("\n" + "=" * 70)
    print("VALIDASI / CONSISTENCY CHECK")
    print("=" * 70)
    for name, ok, detail in checks:
        print(f"[{'OK' if ok else 'GAGAL'}] {name}" + (f"  ({detail})" if detail else ""))
    if not all(c[1] for c in checks):
        print("\n>>> ADA PEMERIKSAAN YANG GAGAL — periksa sebelum memakai hasil untuk laporan. <<<")
    return all(c[1] for c in checks)


def print_summary(df, ev, t_overall, t_period):
    print("\n" + "=" * 70)
    print("RINGKASAN")
    print("=" * 70)
    print("DATASET")
    print(f"  n rows   : {len(df)}")
    print(f"  date min : {df['Date'].min():%Y-%m-%d}")
    print(f"  date max : {df['Date'].max():%Y-%m-%d}")
    for var, title in (("Return", "RETURN (proporsi)"), ("Volume", "VOLUME (lot)")):
        r = t_overall.loc[var]
        print(f"\n{title}")
        print(f"  n        : {int(r['n'])}")
        print(f"  mean     : {r['mean']:.6f}")
        print(f"  median   : {r['median']:.6f}")
        print(f"  SD       : {r['sd']:.6f}")
        print(f"  min      : {r['min']:.6f}")
        print(f"  max      : {r['max']:.6f}")
        print(f"  skewness : {r['skewness']:.4f}")
        print(f"  kurtosis : {r['excess_kurtosis']:.4f}  (excess)")
    print("\nPERIODE")
    for p in PERIOD_ORDER:
        print(f"  {p:<7} n : {int(t_period.loc[p, 'n_obs'])}")
    print("\nEVENT")
    print(f"  jumlah event : {len(ev)}")


# ==========================================================================
# 15. OUTPUT
# ==========================================================================
def save_csv(t, name, index=True):
    path = OUT_DIR / name
    t.to_csv(path, index=index, encoding="utf-8-sig")   # presisi penuh, tanpa pembulatan
    print("  tersimpan:", path)


def main():
    pd.set_option("display.width", 220)
    pd.set_option("display.max_columns", 50)
    pd.set_option("display.float_format", lambda v: f"{v:,.6f}")

    # 01
    raw_data, raw_event = load_data(INPUT_FILE)
    # 02
    df, ev = prepare_data(raw_data, raw_event)

    # 03-06 tabel
    print("\n[03] Statistik deskriptif keseluruhan")
    t_overall = table_overall(df)
    print(t_overall)
    save_csv(t_overall, "tabel_statistik_deskriptif.csv")

    print("\n[04] Statistik deskriptif per periode")
    t_period = table_period(df)
    print(t_period)
    save_csv(t_period, "tabel_deskriptif_periode.csv")

    print("\n[05] Statistik deskriptif rumor / non-rumor "
          "(hanya observasi berlabel: Pre, Event, Post; periode Normal tidak berlabel)")
    t_rumor = table_rumor(df)
    if t_rumor is not None:
        print(t_rumor)
        save_csv(t_rumor, "tabel_deskriptif_rumor.csv")

    print("\n[06] Per event")
    t_event = table_per_event(df, ev)
    print(t_event.to_string())
    save_csv(t_event, "tabel_per_event.csv", index=False)

    # 07-13 grafik
    print("\n[07-13] Visualisasi")
    fig_time_series(df, ev)
    fig_hist_return(df)
    fig_hist_volume(df)
    fig_hist_logvolume(df)
    boxplot_by_period(df, "Return", 100, "Return harian (%)",
                      "Boxplot Return Harian BYAN per Periode",
                      "fig_4_5_boxplot_return_periode.png", "box_return", zero_line=True)
    boxplot_by_period(df, "logVol", 1, "ln(Volume)  [Volume dalam lot]",
                      "Boxplot ln(Volume) Harian BYAN per Periode",
                      "fig_4_6_boxplot_logvolume_periode.png", "box_logvol")
    fig_summary(t_period)
    if MAKE_EVENT_FIGURE:
        fig_event_ratio(t_event)

    # 14 validasi + ringkasan
    validate(df, ev, raw_data, raw_event, t_overall, t_period, t_rumor, t_event)
    print_summary(df, ev, t_overall, t_period)

    # 15 Excel opsional
    try:
        xlsx = OUT_DIR / "hasil_statistik_deskriptif.xlsx"
        with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
            t_overall.to_excel(w, sheet_name="Deskriptif_Keseluruhan")
            t_period.to_excel(w, sheet_name="Deskriptif_Periode")
            if t_rumor is not None:
                t_rumor.to_excel(w, sheet_name="Deskriptif_Rumor")
            t_event.to_excel(w, sheet_name="Per_Event", index=False)
        print("\n  tersimpan:", xlsx)
    except Exception as e:  # noqa
        print("\nExcel ringkasan dilewati:", e)


if __name__ == "__main__":
    main()

Sheet pada workbook: ['BYAN_Analysis', 'Event_Rumor', 'Parameter', 'Skala_Variabel']

SHEET: BYAN_Analysis
shape : (650, 12)
kolom : ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Return', '%ΔVolume', 'Event', 'Event_ID', 'Periode_Event', 'Rumor_Event']

5 baris pertama:
        Date   Open   High    Low  Close  Volume    Return  %ΔVolume  Event Event_ID Periode_Event Rumor_Event
0 2024-01-02  18869  18988  18702  18988   53800       NaN       NaN      0        -        Normal           -
1 2024-01-03  18988  18988  18655  18655   56100 -0.017537  0.042751      0        -        Normal           -
2 2024-01-04  18678  18678  18321  18583  112500 -0.003860  1.005348      0        -        Normal           -
3 2024-01-05  18631  18964  18559  18940  130600  0.019211  0.160889      0        -        Normal           -
4 2024-01-08  19035  19035  18607  18821   60900 -0.006283 -0.533691      0        -        Normal           -

Tipe data:
Date             datetime64[ns]
Open         